# 00datacheck: what data do we have for Kita?

PF7, "Kita: Malaysia in 3.2 million dots". This notebook downloads every source once into `data/raw`, then checks each one before any design work:

1. Census by district (DOSM, 1970 to 2020): population, households, ethnicity, age, urban and rural
2. Population by district (OpenDOSM, 2020 to 2024): sex, five-year age groups, ethnicity
3. Household income by district (OpenDOSM, HIES): mean and median, plus Gini and poverty for 2022
4. Household income percentiles by state (OpenDOSM): for the "Where do you stand?" step
5. District boundaries (DOSM, 160 districts)
6. Kontur population grid (400 m H3 hexagons): to place the dots inside each district

At the end it writes `data/datacheck_summary.json`, which the later notebooks and the case study can quote.

## 0. Settings

In [1]:
from pathlib import Path
import json
import gzip
import requests
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd()                                   # <- run from the PF7 folder
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

KONTUR_FILE = "kontur_population_MY_20231101.gpkg"   # <- Kontur population for Malaysia, 400 m H3 hexagons (CC BY 4.0, listed on data.humdata.org)
KONTUR_URL = f"https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/{KONTUR_FILE}.gz"
CENSUS_YEAR = 2020                                  # <- census year used for the dots
PEOPLE_PER_DOT = 10                                 # <- 1 dot per 10 people

GITHUB = "https://raw.githubusercontent.com/dosm-malaysia/data-open/main/datasets"
SOURCES = {
    "census_district.csv": f"{GITHUB}/census/census_district.csv",
    "administrative_2_district.geojson": f"{GITHUB}/geodata/administrative_2_district.geojson",
    "administrative_1_state.geojson": f"{GITHUB}/geodata/administrative_1_state.geojson",
    "population_district.parquet": "https://storage.dosm.gov.my/population/population_district.parquet",
    "hh_income_district.parquet": "https://storage.dosm.gov.my/hies/hh_income_district.parquet",
    "hies_district.parquet": "https://storage.dosm.gov.my/hies/hies_district.parquet",
    "hies_state_percentile.parquet": "https://storage.dosm.gov.my/hies/hies_state_percentile.parquet",
}
HEADERS = {"User-Agent": "Kita portfolio project (data check)"}
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 1. Download (once)

Files already in `data/raw` are not downloaded again. Delete a file to refresh it.

In [2]:
status = {}
for name, url in SOURCES.items():
    path = RAW / name
    if path.exists() and path.stat().st_size > 0:
        status[name] = "cached"
        continue
    try:
        r = requests.get(url, headers=HEADERS, timeout=120)
        r.raise_for_status()
        path.write_bytes(r.content)
        status[name] = "downloaded"
    except Exception as e:
        status[name] = f"FAILED: {e}"

kontur_path = RAW / KONTUR_FILE
if kontur_path.exists() and kontur_path.stat().st_size > 0:
    status[KONTUR_FILE] = "cached"
else:
    r = requests.get(KONTUR_URL, headers=HEADERS, timeout=600)
    r.raise_for_status()
    kontur_path.write_bytes(gzip.decompress(r.content))
    status[KONTUR_FILE] = "downloaded"

for name, s in status.items():
    size = (RAW / name).stat().st_size / 1e6 if (RAW / name).exists() else 0
    print(f"{name:40s} {s:20s} {size:8.2f} MB")

census_district.csv                      cached                   0.17 MB
administrative_2_district.geojson        cached                   0.95 MB
administrative_1_state.geojson           cached                   0.39 MB
population_district.parquet              cached                   0.26 MB
hh_income_district.parquet               cached                   0.01 MB
hies_district.parquet                    cached                   0.01 MB
hies_state_percentile.parquet            cached                   0.06 MB
kontur_population_MY_20231101.gpkg       cached                  29.11 MB


## 2. Census by district (1970 to 2020)

The DOSM census file has one row per district per census year. Two things to check: that 2020 adds up to the census total (32.4 million), and what the ethnicity columns cover (citizens only, or everyone).

In [3]:
census = pd.read_csv(RAW / "census_district.csv")
census = census.loc[:, ~census.columns.str.startswith("Unnamed")]
print(census.shape)
print(census["year"].value_counts().sort_index().to_dict())
c20 = census[census["year"] == CENSUS_YEAR].copy()
num_cols = [c for c in c20.columns if c not in ["state", "district", "code_state_district"]]
c20[num_cols] = c20[num_cols].apply(pd.to_numeric, errors="coerce")
print("districts:", len(c20))

(960, 40)
{1970: 160, 1980: 160, 1991: 160, 2000: 160, 2010: 160, 2020: 160}
districts: 160


In [4]:
total = c20["population_total"].sum()
citizens = c20["nationality_citizen"].sum()
noncit = c20["nationality_non_citizen"].sum()
eth = c20[["ethnicity_bumi", "ethnicity_chinese", "ethnicity_indian", "ethnicity_other"]].sum()
age = c20[["age_0_14", "age_15_64", "age_65_above"]].sum()
print(f"population {total:,.0f}")
print(f"citizens {citizens:,.0f} + non-citizens {noncit:,.0f} = {citizens + noncit:,.0f}")
print(f"ethnicity sum {eth.sum():,.0f} (equals citizens? {abs(eth.sum() - citizens) < 1000})")
print(f"age sum {age.sum():,.0f}")
print(eth.round(0).to_string())

population 32,447,385
citizens 29,756,315 + non-citizens 2,691,070 = 32,447,385
ethnicity sum 29,756,315 (equals citizens? True)
age sum 32,447,385
ethnicity_bumi       20649533.0
ethnicity_chinese     6892367.0
ethnicity_indian      1998778.0
ethnicity_other        215637.0


### Urban and rural

Some districts have an empty urban or rural cell. If urban + rural still equals the total when the empty cell is read as zero, the empty cell means "none".

In [5]:
ur = c20[["state", "district", "population_total", "density_urban", "density_rural"]].copy()
ur["urban0"] = ur["density_urban"].fillna(0)
ur["rural0"] = ur["density_rural"].fillna(0)
ur["gap"] = ur["population_total"] - ur["urban0"] - ur["rural0"]
print("districts with an empty urban cell:", ur["density_urban"].isna().sum())
print("districts with an empty rural cell:", ur["density_rural"].isna().sum())
print("largest gaps after reading empty as zero:")
print(ur.reindex(ur["gap"].abs().sort_values(ascending=False).index).head(8).to_string(index=False))
print(f"urban share {ur['urban0'].sum() / total:.1%}")

districts with an empty urban cell: 45
districts with an empty rural cell: 6
largest gaps after reading empty as zero:
state    district  population_total  density_urban  density_rural    urban0   rural0  gap
Johor  Batu Pahat          495338.0       309750.0       185588.0  309750.0 185588.0  0.0
Johor Johor Bahru         1711191.0      1702424.0         8767.0 1702424.0   8767.0  0.0
Johor      Kluang          323762.0       225555.0        98207.0  225555.0  98207.0  0.0
Johor Kota Tinggi          222382.0       107232.0       115150.0  107232.0 115150.0  0.0
Johor     Mersing           78195.0        22553.0        55642.0   22553.0  55642.0  0.0
Johor        Muar          314776.0       180241.0       134535.0  180241.0 134535.0  0.0
Johor     Pontian          173318.0        83930.0        89388.0   83930.0  89388.0  0.0
Johor     Segamat          197762.0       104669.0        93093.0  104669.0  93093.0  0.0
urban share 75.1%


In [6]:
hh = c20[["household_total", "household_size_avg"]]
print(f"households {hh['household_total'].sum():,.0f}")
print(hh.describe().round(2).to_string())

households 8,234,644
       household_total  household_size_avg
count           160.00              160.00
mean          51466.52                4.04
std           87688.63                0.53
min            2123.00                2.80
25%           11101.25                3.70
50%           24913.00                4.00
75%           53361.00                4.30
max          617686.00                5.80


## 3. Population by district (2020 to 2024)

Five-year age groups, sex and ethnicity, with non-citizens as their own group. Check the unit (DOSM often publishes thousands) and whether the districts match the census file.

In [7]:
pop_path = RAW / "population_district.parquet"
if pop_path.exists():
    pop = pd.read_parquet(pop_path)
    print(pop.shape)
    print(pop.dtypes.to_string())
    print(pop.head(3).to_string())
    for col in ["date", "sex", "age", "ethnicity"]:
        if col in pop.columns:
            print(col, sorted(pop[col].astype(str).unique())[:25])
else:
    pop = None
    print("population_district.parquet missing: see the download step")

(451668, 7)
state             str
district          str
date           object
sex               str
age               str
ethnicity         str
population    float64
   state    district        date   sex      age   ethnicity  population
0  Johor  Batu Pahat  2020-01-01  both  overall     overall       495.3
1  Johor  Batu Pahat  2020-01-01  both  overall  bumi_malay       311.3
2  Johor  Batu Pahat  2020-01-01  both  overall  bumi_other         5.1


date ['2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01', '2024-01-01', '2025-01-01', '2026-01-01']
sex ['both', 'female', 'male']
age ['0-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80-84', '85+', 'overall']
ethnicity ['bumi_malay', 'bumi_other', 'chinese', 'indian', 'other_citizen', 'other_noncitizen', 'overall']


In [8]:
if pop is not None:
    pop["year"] = pd.to_datetime(pop["date"]).dt.year
    allrows = pop[(pop["sex"] == "both") & (pop["age"] == "overall") & (pop["ethnicity"] == "overall")]
    by_year = allrows.groupby("year")["population"].sum()
    print("national total by year (raw unit):")
    print(by_year.to_string())
    print("districts per year:", allrows.groupby("year")["district"].nunique().to_dict())

national total by year (raw unit):
year
2020    32447.1
2021    32576.0
2022    32698.4
2023    33401.9
2024    34051.9
2025    34220.4
2026    34389.6
districts per year: {2020: 160, 2021: 160, 2022: 160, 2023: 160, 2024: 160, 2025: 166, 2026: 166}


## 4. Household income (HIES)

District mean and median by survey year, the 2022 district table with Gini and poverty, and the state percentile table for the "Where do you stand?" step.

In [9]:
inc = {}
for name in ["hh_income_district.parquet", "hies_district.parquet", "hies_state_percentile.parquet"]:
    path = RAW / name
    if not path.exists():
        print(name, "missing")
        continue
    df = pd.read_parquet(path)
    inc[name] = df
    print("=" * 70)
    print(name, df.shape)
    print(df.head(3).to_string())
    if "date" in df.columns:
        print("years:", sorted(pd.to_datetime(df["date"]).dt.year.unique().tolist()))
    if "district" in df.columns:
        print("districts:", df["district"].nunique())

hh_income_district.parquet (480, 5)
   state    district        date  income_mean  income_median
0  Johor  Batu Pahat  2019-01-01         7392           6504
1  Johor  Batu Pahat  2022-01-01         7419           6347
2  Johor  Batu Pahat  2024-01-01         9006           7555
years: [2019, 2022, 2024]
districts: 172
hies_district.parquet (322, 8)
         date  state     district  income_mean  income_median  expenditure_mean   gini  poverty
0  2022-01-01  Johor   Batu Pahat         7419           6347              4570  0.338      5.1
1  2024-01-01  Johor   Batu Pahat         9006           7555              5153  0.316      2.4
2  2022-01-01  Johor  Johor Bahru         9869           8232              6139  0.358      3.7
years: [2022, 2024]
districts: 172
hies_state_percentile.parquet (19200, 5)
         date  state  percentile variable  income
0  2024-01-01  Johor           1     mean  1638.0
1  2024-01-01  Johor           1   median  1682.0
2  2024-01-01  Johor           1  mini

In [10]:
if "hh_income_district.parquet" in inc:
    d = inc["hh_income_district.parquet"].copy()
    d["year"] = pd.to_datetime(d["date"]).dt.year
    latest = d[d["year"] == d["year"].max()]
    print("latest year:", d["year"].max(), "districts:", latest["district"].nunique())
    print(latest["income_median"].describe().round(0).to_string())
    print("lowest 5:")
    print(latest.nsmallest(5, "income_median")[["state", "district", "income_median"]].to_string(index=False))
    print("highest 5:")
    print(latest.nlargest(5, "income_median")[["state", "district", "income_median"]].to_string(index=False))

latest year: 2024 districts: 162
count      162.0
mean      5067.0
std       1748.0
min       2525.0
25%       3957.0
50%       4594.0
75%       5799.0
max      11404.0
lowest 5:


  state     district  income_median
Sarawak        Pantu           2525
Sarawak Bukit Mabong           2589
Sarawak      Sebuyau           2717
  Sabah        Pitas           2785
  Sabah  Kota Marudu           2877
highest 5:
   state     district  income_median
Selangor   Ulu Langat          11404
Selangor       Sepang          11131
Selangor       Gombak          10777
Selangor     Petaling          10688
Selangor Kuala Langat          10583


In [11]:
if "hies_state_percentile.parquet" in inc:
    p = inc["hies_state_percentile.parquet"].copy()
    p["year"] = pd.to_datetime(p["date"]).dt.year
    print("years:", sorted(p["year"].unique().tolist()))
    print("states:", sorted(p["state"].unique().tolist()))
    print("variables:", sorted(p["variable"].unique().tolist()))
    print("percentiles:", p["percentile"].min(), "to", p["percentile"].max())

years: [2019, 2022, 2024]
states: ['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']
variables: ['maximum', 'mean', 'median', 'minimum']
percentiles: 1 to 100


## 5. District boundaries and name matching

Every table must join to the 160 district shapes. Names are compared after trimming and lower-casing.

In [12]:
districts = gpd.read_file(RAW / "administrative_2_district.geojson")
print(districts.shape, districts.crs)
print(districts.head(3).drop(columns="geometry").to_string())


def key(df):
    return set((df["state"].str.strip().str.lower() + " | " + df["district"].str.strip().str.lower()).unique())


shape_keys = key(districts)
tables = {"census 2020": c20}
if pop is not None:
    tables["population_district"] = pop
for name, df in inc.items():
    if "district" in df.columns:
        tables[name] = df
for name, df in tables.items():
    k = key(df)
    print(f"{name:32s} {len(k):4d} districts, not in shapes: {sorted(k - shape_keys)[:6]}, shapes not in table: {len(shape_keys - k)}")

(160, 6) EPSG:4326
   state     district  code_state  code_district code_state_district
0  Johor   Batu Pahat           1              1                 1_1
1  Johor  Johor Bahru           1              2                 1_2
2  Johor       Kluang           1              3                 1_3
census 2020                       160 districts, not in shapes: [], shapes not in table: 0


population_district               166 districts, not in shapes: ['sabah | membakut', 'sarawak | gedong', 'sarawak | lingga', 'sarawak | pantu', 'sarawak | sebuyau', 'sarawak | siburan'], shapes not in table: 0
hh_income_district.parquet        171 districts, not in shapes: ['perak | larut & matang', 'pulau pinang | s.p. selatan', 'pulau pinang | s.p.tengah', 'pulau pinang | s.p.utara', 'sabah | membakut', 'sarawak | gedong'], shapes not in table: 0
hies_district.parquet             171 districts, not in shapes: ['perak | larut & matang', 'pulau pinang | s.p. selatan', 'pulau pinang | s.p.tengah', 'pulau pinang | s.p.utara', 'sabah | membakut', 'sarawak | gedong'], shapes not in table: 0


## 6. Kontur population grid

The grid places the dots. Check its total against the census and how much of it falls inside the district shapes. District names are compared in lower case, because the census writes "Larut dan Matang" and the shapes "Larut Dan Matang".

In [13]:
kontur = gpd.read_file(RAW / KONTUR_FILE)
print(kontur.shape, kontur.crs)
print(kontur.head(3).drop(columns="geometry").to_string())
print(f"Kontur total {kontur['population'].sum():,.0f}")

(147936, 3) EPSG:3857
                h3  population
0  888c96db6dfffff         5.0
1  888c96db6bfffff         5.0
2  888c96db65fffff         5.0
Kontur total 34,391,363


In [14]:
cent = kontur.copy()
cent["geometry"] = cent.geometry.centroid
cent = cent.to_crs(4326)
joined = gpd.sjoin(cent, districts[["state", "district", "geometry"]], how="left", predicate="within")
outside = joined["district"].isna()
print(f"hexagons {len(joined):,}, outside every district {outside.sum():,} ({joined.loc[outside, 'population'].sum():,.0f} people)")
joined["district"] = joined["district"].str.lower()
k_by = joined.groupby(["state", "district"])["population"].sum().rename("kontur")
c20k = c20.assign(district=c20["district"].str.lower())
cmp = c20k.set_index(["state", "district"])["population_total"].rename("census").to_frame().join(k_by)
cmp["ratio"] = cmp["kontur"] / cmp["census"]
print("districts with no Kontur hexagon:", cmp["kontur"].isna().sum())
print(cmp["ratio"].describe().round(2).to_string())
print("largest differences:")
print(cmp.reindex((cmp["ratio"] - 1).abs().sort_values(ascending=False).index).head(8).round(2).to_string())

hexagons 147,951, outside every district 3,628 (405,166 people)
districts with no Kontur hexagon: 0
count    160.00
mean       1.10
std        0.31
min        0.08
25%        0.95
50%        1.05
75%        1.19
max        2.48
largest differences:
                                 census   kontur  ratio
state          district                                
Sarawak        tanjung manis     7946.0  19742.0   2.48
               belaga           22502.0  53945.0   2.40
               matu             16316.0  35110.0   2.15
               song              9961.0  21340.0   2.14
Sabah          kalabakan        48195.0  99963.0   2.07
Sarawak        bukit mabong     10155.0  19870.0   1.96
W.P. Putrajaya w.p. putrajaya  109202.0   8587.0   0.08
Sabah          putatan          68811.0   9738.0   0.14


## 7. Dots budget

How many dots at the chosen ratio, and roughly how large the web file would be (longitude, latitude and one byte each for group, age band and urban or rural).

In [15]:
n_dots = int(round(total / PEOPLE_PER_DOT))
bytes_per_dot = 4 + 4 + 3
print(f"{n_dots:,} dots at 1 per {PEOPLE_PER_DOT}")
print(f"about {n_dots * bytes_per_dot / 1e6:.0f} MB uncompressed")

3,244,738 dots at 1 per 10
about 36 MB uncompressed


## 8. Summary file

In [16]:
summary = {
    "census_year": CENSUS_YEAR,
    "population_total": int(total),
    "citizens": int(citizens),
    "non_citizens": int(noncit),
    "ethnicity_citizens": {k: int(v) for k, v in eth.items()},
    "households": int(c20["household_total"].sum()),
    "urban_share": round(float(ur["urban0"].sum() / total), 4),
    "districts": int(len(c20)),
    "kontur_total": int(kontur["population"].sum()),
    "people_per_dot": PEOPLE_PER_DOT,
    "n_dots": n_dots,
    "downloads": status,
}
(ROOT / "data" / "datacheck_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "census_year": 2020,
  "population_total": 32447385,
  "citizens": 29756315,
  "non_citizens": 2691070,
  "ethnicity_citizens": {
    "ethnicity_bumi": 20649533,
    "ethnicity_chinese": 6892367,
    "ethnicity_indian": 1998778,
    "ethnicity_other": 215637
  },
  "households": 8234644,
  "urban_share": 0.7506,
  "districts": 160,
  "kontur_total": 34391363,
  "people_per_dot": 10,
  "n_dots": 3244738,
  "downloads": {
    "census_district.csv": "cached",
    "administrative_2_district.geojson": "cached",
    "administrative_1_state.geojson": "cached",
    "population_district.parquet": "cached",
    "hh_income_district.parquet": "cached",
    "hies_district.parquet": "cached",
    "hies_state_percentile.parquet": "cached",
    "kontur_population_MY_20231101.gpkg": "cached"
  }
}


___

**END**